In [1]:
print("hello")

hello


In [2]:
import numpy as np
import torch
import torch.nn as nn


In [3]:
class Dataset:
    def __init__(self):
        pass
    def __len__(self):
        pass
    def __getitem__(self, key):
        pass

dataloader = Dataset
        



In [ ]:
class MultimodalModel(nn.Module):
    def __init__(self, text_model, visual_model):
        self.text_model = text_model
        self.visual_model = visual_model
                # Freeze pretrained encoders
        for param in self.text_model.parameters():
            param.requires_grad = False

        for param in self.video_model.parameters():
            param.requires_grad = False
        
        self.classfier = nn.Sequential(
            nn.Linear(768+768, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 7)
        )

    def forward(self, text, visual):
        text = self.text_model(text)
        visual = self.visual_model(visual)

        combined = torch.cat([text, visual], dim=-1)
        logits = self.classfier(combined)
        return logits

In [ ]:
#Training loop
from transformers import (
    AutoTokenizer,
    AutoModel,
    VideoMAEImageProcessor,
    VideoMAEModel,
)

text_model = AutoModel.from_pretrained(
    "microsoft/deberta-v3-small"
)

video_model = VideoMAEModel.from_pretrained(
    "MCG-NJU/videomae-base"
)

model = MultimodalModel(
    text_model=text_model,
    visual_model=video_model
)

num_epochs = 10
learning_rate = 2e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    model.train()
    for batch in dataloader:
        optimizer.zero_grad()
        logits = model(
            batch["text"],
            batch["visual"]
        )
        labels = batch[labels]
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()